In [2]:
# Prepare the Data for Machine Learning Algorithms
import pandas as pd
import numpy as np
from utils import load_housing_data
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

housing = load_housing_data()
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)

strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42
)

# Revert to a clean training set
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

print(housing.info())

<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 13096 to 19888
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           16512 non-null  float64 
 1   latitude            16512 non-null  float64 
 2   housing_median_age  16512 non-null  float64 
 3   total_rooms         16512 non-null  float64 
 4   total_bedrooms      16344 non-null  float64 
 5   population          16512 non-null  float64 
 6   households          16512 non-null  float64 
 7   median_income       16512 non-null  float64 
 8   ocean_proximity     16512 non-null  object  
 9   income_cat          16512 non-null  category
dtypes: category(1), float64(8), object(1)
memory usage: 1.3+ MB
None


### Clean the Data (total_bedrooms attribute has some missing values.)
1. Get rid of the corresponding districts.
2. Get rid of the whole attribute.
3. Set the missing values to some value (zero, the mean, the median, etc.). This is
called imputation.

In [15]:
housing.dropna(subset=["total_bedrooms"], inplace=True) # option 1
print(housing.info())

<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 13096 to 19888
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           16512 non-null  float64 
 1   latitude            16512 non-null  float64 
 2   housing_median_age  16512 non-null  float64 
 3   total_rooms         16512 non-null  float64 
 4   total_bedrooms      16512 non-null  float64 
 5   population          16512 non-null  float64 
 6   households          16512 non-null  float64 
 7   median_income       16512 non-null  float64 
 8   ocean_proximity     16512 non-null  object  
 9   income_cat          16512 non-null  category
dtypes: category(1), float64(8), object(1)
memory usage: 1.3+ MB
None


In [16]:
housing.drop("total_bedrooms", axis=1, inplace=True) # option 2
print(housing.info())

<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 13096 to 19888
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           16512 non-null  float64 
 1   latitude            16512 non-null  float64 
 2   housing_median_age  16512 non-null  float64 
 3   total_rooms         16512 non-null  float64 
 4   population          16512 non-null  float64 
 5   households          16512 non-null  float64 
 6   median_income       16512 non-null  float64 
 7   ocean_proximity     16512 non-null  object  
 8   income_cat          16512 non-null  category
dtypes: category(1), float64(7), object(1)
memory usage: 1.1+ MB
None


In [20]:
median = housing["total_bedrooms"].median() # option 3
housing["total_bedrooms"] = housing["total_bedrooms"].fillna(median)
print(housing.info())

<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 13096 to 19888
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   longitude           16512 non-null  float64 
 1   latitude            16512 non-null  float64 
 2   housing_median_age  16512 non-null  float64 
 3   total_rooms         16512 non-null  float64 
 4   total_bedrooms      16512 non-null  float64 
 5   population          16512 non-null  float64 
 6   households          16512 non-null  float64 
 7   median_income       16512 non-null  float64 
 8   ocean_proximity     16512 non-null  object  
 9   income_cat          16512 non-null  category
dtypes: category(1), float64(8), object(1)
memory usage: 1.3+ MB
None


### Clean the Data (imputation.) with Sklearn

In [3]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
# Copy a data only the numerical atts, exclude ocean_proximity
housing_num = housing.select_dtypes(include=[np.number])
# Calculates the median for each column in housing_num
imputer.fit(housing_num)
# Check calculated medians
print(housing_num.median().values)
# transform replaces gaps with calculated medians
X = imputer.transform(housing_num)

# recover the column names and index from housing_num
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)
print(housing_tr.info())


[-118.51     34.26     29.     2125.      434.     1167.      408.
    3.5385]
<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 13096 to 19888
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           16512 non-null  float64
 1   latitude            16512 non-null  float64
 2   housing_median_age  16512 non-null  float64
 3   total_rooms         16512 non-null  float64
 4   total_bedrooms      16512 non-null  float64
 5   population          16512 non-null  float64
 6   households          16512 non-null  float64
 7   median_income       16512 non-null  float64
dtypes: float64(8)
memory usage: 1.1 MB
None


In [ ]:
# Format text data to numerical data

# Read original data
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(8)

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

print(housing_cat_encoded[:8])
print(housing_cat[:8])
print(ordinal_encoder.categories_)

# '<1H OCEAN' < 'INLAND' < 'ISLAND' < 'NEAR BAY' < 'NEAR OCEAN'
# 0 < 1 < 2 < 3 < 4
# NEAR OCEAN (4) >  INLAND (1) !!!
# "NEAR THE OCEAN is 4 times farther from <1H OCEAN than INLAND"
# problem can be in linear regression 
# price = w * ocean_proximity
# The higher the number, the higher the price. 
# NEAR OCEAN (4) is always "better" than INLAND (1).



[[3.]
 [0.]
 [1.]
 [1.]
 [4.]
 [1.]
 [0.]
 [3.]]
      ocean_proximity
13096        NEAR BAY
14973       <1H OCEAN
3785           INLAND
14689          INLAND
20507      NEAR OCEAN
1286           INLAND
18078       <1H OCEAN
4396         NEAR BAY
[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
      dtype=object)]


In [79]:
# Can solve promblem with OneHotEncoder
cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
# Categories are equal,  There is no automatic “better/worse”
print(housing_cat_1hot[:8].toarray())
# price = w1*INLAND + w2*NEAR_BAY + w3*ISLAND + ...
print("Sparse format: (row, column) -> value\n", housing_cat_1hot[:8])
# list of cats encoder
cat_encoder.categories_

[[0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]]
Sparse format: (row, column) -> value
   (0, 3)	1.0
  (1, 0)	1.0
  (2, 1)	1.0
  (3, 1)	1.0
  (4, 4)	1.0
  (5, 1)	1.0
  (6, 0)	1.0
  (7, 3)	1.0


[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [80]:
# Pandas function to convert categorical feature into one-hot representation
df_test = pd.DataFrame({"ocean_proximity": ["INLAND", "NEAR BAY"]})
pd.get_dummies(df_test)

# Using trained OneHotEncoder (fit on train data before)
result = cat_encoder.transform(df_test)
print("trained OneHotEncoder\n")
print(result[:8].toarray())

# Do NOT use pd.get_dummies() in ML pipelines:
# - It creates columns based only on current data
# - Train and test may have different categories → different columns
# - This leads to feature mismatch and breaks the model

# OneHotEncoder:
# - Learns categories during fit() on training data
# - Always outputs the same number and order of features
# - Ensures consistency between train and test data


# Example with unknown category
df_test_unknown = pd.DataFrame({"ocean_proximity": ["<2H OCEAN", "ISLAND"]})
print("example with unknown category\n")
print(pd.get_dummies(df_test_unknown))

# ⚠️ OneHotEncoder will raise an error for unseen categories by default
# This is GOOD: it prevents silent bugs and unexpected behavior

# Optional (use carefully):
#cat_encoder.handle_unknown = "ignore"
# print(cat_encoder.transform(df_test_unknown)[:8].toarray())
# → unknown category will be encoded as [0, 0, ..., 0]

# ⚠️ Warning:
# Ignoring unknown categories may hide data issues
# Use only if you understand the implications

trained OneHotEncoder

[[0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0.]]
example with unknown category

   ocean_proximity_<2H OCEAN  ocean_proximity_ISLAND
0                       True                   False
1                      False                    True


In [81]:
# Avoiding column mistmatches and useful when debugging
print(cat_encoder.feature_names_in_)

print(cat_encoder.get_feature_names_out())

# Found unknown categories ['<2H OCEAN'] in column 0 during transform
df_output = pd.DataFrame(
    cat_encoder.transform(df_test_unknown),
    columns=cat_encoder.get_feature_names_out(),
    index=df_test_unknown.index,
)

['ocean_proximity']
['ocean_proximity_<1H OCEAN' 'ocean_proximity_INLAND'
 'ocean_proximity_ISLAND' 'ocean_proximity_NEAR BAY'
 'ocean_proximity_NEAR OCEAN']


ValueError: Found unknown categories ['<2H OCEAN'] in column 0 during transform